# 00 - GPU smoke test

Confirms that the instance's GPUs are visible, that PyTorch can use them, and
roughly what throughput they deliver. Run every cell top to bottom.

`02-jupyter.sh` copies this notebook to the instance's home directory. It lives on
the EBS volume, which is deleted with the instance.

This checks one GPU at a time. For the multi-GPU DDP and NCCL AllReduce numbers,
run `make bench` from your laptop, which drives `bench/ddp_allreduce.py` under
`torchrun` across every GPU on the instance.


## 1. Driver and device, straight from the node

In [ ]:
!nvidia-smi

In [ ]:
# Topology matrix. On g4dn.12xlarge the four T4s hang off PCIe with no NVLink,
# so expect PHB or NODE between every pair rather than NV#. That is the ceiling
# the AllReduce bandwidth in `make bench` runs into.
!nvidia-smi topo -m


## 2. PyTorch sees the GPU

In [ ]:
import torch

print('torch          :', torch.__version__)
print('cuda available :', torch.cuda.is_available())
print('cuda runtime   :', torch.version.cuda)
print('device count   :', torch.cuda.device_count())

assert torch.cuda.is_available(), 'No CUDA device. Check nvidia-smi over SSH.'

# g4dn.12xlarge has four of these. They are identical, but print each one so a
# partially-visible GPU (a stuck driver on one device) is obvious rather than
# silently halving the benchmark.
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print()
    print('cuda:%d' % i)
    print('  name         :', props.name)
    print('  capability   : sm_%d%d' % (props.major, props.minor))
    print('  total memory : %.1f GiB' % (props.total_memory / 1024**3))
    print('  SM count     :', props.multi_processor_count)

print()
print('bf16 supported :', torch.cuda.is_bf16_supported())


## 3. Matmul throughput

A crude but honest check that the GPU is actually doing work.

The dtype is chosen from the hardware. The **T4** in a `g4dn` is sm_75 and has no
bf16, so this falls back to fp16 — same tensor-core throughput on that generation,
~65 TFLOP/s dense peak. On an **L4** (`g6`) it uses bf16; the L4's headline 121
TFLOP/s is the *sparse* number needing 2:4 structured sparsity, and dense peak is
about half that, ~60 TFLOP/s. A previous single-L4 run measured **56.2 TFLOP/s**,
roughly 93% of dense peak.

Anything near 1 TFLOP/s means you are silently running on CPU.


In [ ]:
import time
import torch

# The T4 has no bf16, and autocasting to it there measures a software fallback
# rather than the GPU. fp16 is the right 16-bit type on that generation.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('dtype:', dtype)

n, iters = 8192, 50
a = torch.randn(n, n, device='cuda', dtype=dtype)
b = torch.randn(n, n, device='cuda', dtype=dtype)

for _ in range(10):          # warm up: first calls include kernel autotuning
    a @ b
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(iters):
    a @ b
torch.cuda.synchronize()
elapsed = time.perf_counter() - start

flops = 2 * n**3 * iters     # one multiply-add per output element per k
print('%.1f ms/matmul' % (elapsed / iters * 1e3))
print('%.1f TFLOP/s %s' % (flops / elapsed / 1e12, str(dtype).replace('torch.', '')))


## 4. NCCL initialises

Single process, single rank, so this measures nothing about the interconnect. It
only proves the NCCL library loads and the process group forms, which is the thing
that breaks first when you scale out. `make bench` is what actually exercises the
four-GPU AllReduce.


In [ ]:
import os
import torch
import torch.distributed as dist

os.environ.setdefault('MASTER_ADDR', '127.0.0.1')
os.environ.setdefault('MASTER_PORT', '29500')
os.environ.setdefault('RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')

if not dist.is_initialized():
    dist.init_process_group(backend='nccl')

torch.cuda.set_device(0)
t = torch.ones(1024, device='cuda')
dist.all_reduce(t)
torch.cuda.synchronize()

print('world size :', dist.get_world_size())
print('all_reduce :', t[0].item(), '(expected 1.0 at world size 1)')

dist.destroy_process_group()

## 5. What the instance is

Confirms the instance really is the type you asked for - a Spot request that fell
back to a different size would show up here.


In [ ]:
import socket
import subprocess

# IMDSv2: the Deep Learning AMI enforces the token handshake, so a plain GET of
# 169.254.169.254 returns 401 and looks like a hung network call.
IMDS = """
TOKEN=$(curl -s -X PUT http://169.254.169.254/latest/api/token \
  -H 'X-aws-ec2-metadata-token-ttl-seconds: 60')
curl -s -H "X-aws-ec2-metadata-token: $TOKEN" \
  http://169.254.169.254/latest/meta-data/instance-type
"""

def sh(cmd):
    return subprocess.run(['bash', '-lc', cmd], capture_output=True,
                          text=True).stdout.strip()

print('hostname      :', socket.gethostname())
print('instance type :', sh(IMDS) or '(metadata unavailable)')
print('vCPU visible  :', sh('grep -c ^processor /proc/cpuinfo'))
print('RAM visible   :', sh("free -g | awk '/Mem:/ {print $2}'"), 'GiB')
